# Jejak Bara v3.0 — notebook pemrosesan (data asli NASA)

Jalankan dari atas ke bawah. Hasil akhirnya: folder `keluaran/` (dibaca aplikasi web) dan `olahan/` (grafik untuk slide)
di Google Drive. Unduh `keluaran.zip` di sel terakhir, ekstrak menimpa `keluaran/` di repo, commit, push.

**Data NASA yang dipakai**
| Data | Koleksi | Untuk |
|---|---|---|
| NISAR L2 GCOV (HHHH, HVHV) | `NISAR_L2_GCOV_V1` → `_PROVISIONAL_V1` → `_BETA_V1` (ASF DAAC) | deteksi kehilangan tajuk, SLIC + Multi-Otsu |
| VIIRS 375 m active fire | FIRMS `VIIRS_SNPP_SP`, `VIIRS_SNPP_NRT`, `VIIRS_NOAA20_NRT` | label terkonfirmasi, titik bakar mulai, FRP |
| MERRA-2 aerosol | `M2TMNXAER` (GES DISC), `TOTEXTTAU` | muatan asap (AOD) per poligon |
| Citra optik berasap | NASA GIBS / Worldview (dipakai langsung di aplikasi) | konteks "smoke paradox" |

**Sebelum mulai:** unggah `jejak_bara.py` ke Drive `MyDrive/spaceapps_nisar/`. Login sekali ke ASF Vertex di browser untuk
menyetujui EULA NISAR (tanpa itu unduhan 401). Untuk MERRA-2, setujui aplikasi *NASA GESDISC DATA ARCHIVE* di profil Earthdata.

## 1. Pasang paket


In [ ]:
!pip install -q earthaccess rasterio rioxarray geopandas xarray h5netcdf h5py scikit-image netCDF4

## 2. Drive dan modul


In [ ]:
from google.colab import drive
import os, sys, json, glob
drive.mount('/content/drive')
K = '/content/drive/MyDrive/spaceapps_nisar'
for sub in ['mentah', 'olahan', 'keluaran']:
    os.makedirs(f'{K}/{sub}', exist_ok=True)
assert os.path.exists(f'{K}/jejak_bara.py'), 'Unggah jejak_bara.py ke ' + K + ' dulu'
sys.path.insert(0, K)
import importlib, jejak_bara as jb; importlib.reload(jb)
print('modul siap. AOI:', list(jb.AOI_KANDIDAT))


## 3. Login Earthdata
Prompt interaktif — password tidak tersimpan di notebook. Pastikan sudah login sekali ke ASF Vertex untuk menyetujui EULA.


In [ ]:
import earthaccess
auth = earthaccess.login(strategy='interactive', persist=True)
print('terautentikasi:', auth.authenticated)


## 4. AOI yang dipakai
Ubah `bbox` di sini kalau klaster titik panas (sel 6) ternyata di luar kotak. Kalteng dan Merauke: S dan N **negatif**.


In [ ]:
AOI = {k: dict(v) for k, v in jb.AOI_KANDIDAT.items()}
# contoh mempersempit: AOI['riau']['bbox'] = (101.8, 0.6, 102.6, 1.4)
for k, v in AOI.items(): print(f'{k:8s} {v["bbox"]}  {v["nama"]}')


## 5. Titik panas VIIRS per AOI (API area FIRMS, blok 5 hari)
Kunci gratis: `firms.modaps.eosdis.nasa.gov/api/map_key`. Dimasukkan lewat prompt, tidak disimpan di notebook. DAY_RANGE maksimal 5.

In [ ]:
import io, requests, datetime as dt, pandas as pd, getpass, time
MAP_KEY = getpass.getpass('FIRMS MAP_KEY: ').strip()

# 0) cek kunci dulu
st = requests.get(f'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY={MAP_KEY}', timeout=30)
print('status MAP_KEY:', st.status_code, st.text[:200])

SUMBER = ['VIIRS_SNPP_SP', 'VIIRS_SNPP_NRT', 'VIIRS_NOAA20_NRT']   # sains dulu, lalu NRT
T0, T1, LANGKAH = dt.date(2026, 6, 1), dt.date(2026, 11, 14), 5     # DAY_RANGE maksimal 5

def ambil(sumber, bbox, d):
    w, s, e, n = bbox
    url = f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{sumber}/{w},{s},{e},{n}/{LANGKAH}/{d}'
    r = requests.get(url, timeout=120)
    teks = r.text.strip()
    if not r.ok or not teks.startswith('latitude'):          # header CSV FIRMS selalu diawali 'latitude'
        return None, teks[:150]
    return pd.read_csv(io.StringIO(teks)), None

potongan, galat = [], []
for k, v in AOI.items():
    d = T0
    while d <= T1:
        for s in SUMBER:
            df, err = ambil(s, v['bbox'], d)
            if df is not None:
                if len(df):
                    df['sumber'] = s; df['aoi'] = k; potongan.append(df)
                break                     # sumber ini menjawab (meski kosong) -> tak perlu sumber berikutnya
            galat.append(f'{k} {d} {s}: {err}')
        d += dt.timedelta(days=LANGKAH)
        time.sleep(0.2)
    print(f'{k}: selesai, {sum(len(p) for p in potongan if p["aoi"].iloc[0] == k):,} titik')

if galat:
    print(f'\n{len(galat)} permintaan ditolak, contoh:'); print('\n'.join(galat[:5]))
assert potongan, 'Tidak ada data sama sekali — lihat pesan error di atas (MAP_KEY salah? kuota habis?)'

hs = pd.concat(potongan, ignore_index=True).drop_duplicates(['latitude', 'longitude', 'acq_date', 'acq_time'])
HOTSPOT = f'{K}/mentah/hotspot_idn_2026.csv'
hs.to_csv(HOTSPOT, index=False)
print(len(hs), 'titik panas ->', HOTSPOT)
print('confidence:', hs['confidence'].value_counts().to_dict(), '(pipeline hanya memakai n dan h)')
print(hs.groupby(['aoi', 'sumber']).size())

## 6. Sebaran titik panas per AOI
Pastikan kotak AOI menangkap klaster terbesar. Kalau tidak, ubah sel 4 lalu jalankan ulang sel ini.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(AOI), figsize=(5 * len(AOI), 4.5))
for a, (k, v) in zip(ax, AOI.items()):
    w, s, e, n = v['bbox']
    sub = hs[(hs.longitude.between(w, e)) & (hs.latitude.between(s, n))]
    a.scatter(sub.longitude, sub.latitude, s=1, alpha=.3, c=pd.to_datetime(sub.acq_date).map(pd.Timestamp.toordinal), cmap='plasma')
    a.set_xlim(w, e); a.set_ylim(s, n); a.set_title(f'{k}: {len(sub):,} titik panas'); a.set_aspect('equal')
plt.tight_layout(); plt.show()
for k, v in AOI.items():
    w, s, e, n = v['bbox']; sub = hs[(hs.longitude.between(w, e)) & (hs.latitude.between(s, n))]
    if len(sub): print(k, 'puncak bulan:', pd.to_datetime(sub.acq_date).dt.to_period('M').value_counts().idxmax())


## 7. Peta gambut (KHG) dan batas kabupaten per AOI
Urutan sumber KHG: (1) berkas yang **sudah ada** di `mentah/<aoi>/khg.geojson` (data tim), (2) SIPPEG (Peta Fungsi Ekosistem
Gambut 1:50.000, di-dissolve per KHG), domain baru lalu lama, (3) geoportal KLHK lama. Kabupaten selalu dari geoBoundaries,
terpisah, jadi kegagalan KHG tidak menghentikan kabupaten. Kalau semua sumber KHG gagal: unggah berkas KHG tim ke
`MyDrive/spaceapps_nisar/mentah/<aoi>/khg.geojson` lalu jalankan ulang sel ini.


In [ ]:
import geopandas as gpd, requests
SUMBER_KHG = [
    'https://sippeg.kemenlh.go.id/arcgis/rest/services/FEG_50K_Nasional_Agustus_2022/MapServer/0',
    'https://sippeg.menlhk.go.id/arcgis/rest/services/FEG_50K_Nasional_Agustus_2022/MapServer/0',
    'https://geoportal.menlhk.go.id/arcgis/rest/services/KLHK/Kesatuan_Hidrologis_Gambut_/MapServer/0',
]

def query_arcgis(layer_url, bbox):
    fitur, off = [], 0
    while True:
        r = requests.get(f'{layer_url}/query', timeout=180, params={
            'where': '1=1', 'outFields': '*', 'outSR': 4326, 'f': 'geojson',
            'geometry': ','.join(map(str, bbox)), 'geometryType': 'esriGeometryEnvelope', 'inSR': 4326,
            'spatialRel': 'esriSpatialRelIntersects', 'resultOffset': off,
            'maxAllowableOffset': 0.0003, 'geometryPrecision': 5})          # ~30 m, cukup untuk grid 20-80 m
        r.raise_for_status(); j = r.json()
        if 'error' in j: raise RuntimeError(j['error'])
        fitur += j.get('features', [])
        if not (j.get('exceededTransferLimit') or j.get('properties', {}).get('exceededTransferLimit')) or not j.get('features'):
            break
        off = len(fitur)
    return gpd.GeoDataFrame.from_features(fitur, crs='EPSG:4326')

def ambil_khg(k, bbox):
    p = f'{K}/mentah/{k}/khg.geojson'
    if os.path.exists(p):
        g = gpd.read_file(p); return g, 'berkas tim (sudah ada)'
    for url in SUMBER_KHG:
        try:
            g = query_arcgis(url, bbox)
            if len(g) == 0: continue
            kode = next((c for c in g.columns if c.upper() in ('KODE_KHG', 'KD_KHG')), None)
            nama = next((c for c in g.columns if 'KHG' in c.upper() and 'NAMA' in c.upper()), None)
            g['geometry'] = g.geometry.buffer(0)
            if kode:                                    # FEG terpecah lindung/budidaya -> satukan per KHG
                g = g.dissolve(by=kode, aggfunc='first').reset_index()
            g = g[[c for c in [kode, nama] if c] + ['geometry']]
            g.to_file(p, driver='GeoJSON')
            return g, url.split('/')[2]
        except Exception as ex:
            print(f'   {url.split("/")[2]}: gagal ({str(ex)[:90]})')
    return None, None

KOLOM_KAB = 'shapeName'
SUMBER_KAB = ['https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/gbOpen/IDN/ADM2/geoBoundaries-IDN-ADM2_simplified.geojson']
KAB_ALL = None
def kab_semua():
    global KAB_ALL
    if KAB_ALL is None:
        try:
            gb = requests.get('https://www.geoboundaries.org/api/current/gbOpen/IDN/ADM2/', timeout=20).json()
            SUMBER_KAB.insert(0, gb['simplifiedGeometryGeoJSON'])
        except Exception as ex:
            print('   geoboundaries.org tidak merespons, pakai mirror GitHub')
        for u in SUMBER_KAB:
            try:
                KAB_ALL = gpd.read_file(u); break
            except Exception as ex:
                print(f'   {u[:60]}...: gagal ({str(ex)[:80]})')
    return KAB_ALL

for k, v in AOI.items():
    os.makedirs(f'{K}/mentah/{k}', exist_ok=True)
    pk = f'{K}/mentah/{k}/kabupaten.geojson'
    if os.path.exists(pk):
        kab = gpd.read_file(pk)
    else:
        kab = kab_semua().clip(v['bbox']); kab = kab[~kab.is_empty]; kab.to_file(pk, driver='GeoJSON')
    khg, src = ambil_khg(k, v['bbox'])
    if khg is None:
        print(f'{k}: KHG GAGAL dari semua sumber -> unggah berkas KHG ke {K}/mentah/{k}/khg.geojson')
    else:
        print(f'{k}: {len(khg)} poligon KHG ({src}), {len(kab)} kabupaten')


## 7b. Muatan asap (AOD) dari MERRA-2 — opsional, tapi dibutuhkan untuk Pilar 4
Rata-rata bulanan `TOTEXTTAU` (AOD 550 nm) dari `M2TMNXAER` untuk bulan-bulan antara pra dan pasca. Grid 0,5° × 0,625°: kasar,
tapi reanalisis tidak pernah "buta asap" seperti MODIS/VIIRS optik. Kalau lambat, biarkan jalan sambil lanjut ke sel 8.

In [ ]:
TGL_ASAP = ('2026-07-01', '2026-10-31')       # sesuaikan dengan pra/pasca setelah sel 9
m2 = earthaccess.search_data(short_name='M2TMNXAER', temporal=TGL_ASAP)
f_m2 = earthaccess.download(m2, local_path='/content/merra2')
print(len(f_m2), 'berkas MERRA-2')
for k, v in AOI.items():
    jb.buat_aod_rerata(f_m2, v['bbox'], f'{K}/mentah/{k}/aod_merra2.tif', variabel='TOTEXTTAU')

## 8. Cari granule NISAR GCOV per AOI — SEMUA koleksi digabung
V1 (produksi) bisa baru berisi data beberapa minggu terakhir, sementara Juni–Oktober ada di PROVISIONAL/BETA.
Semua koleksi dicari, lalu duplikat dibuang (prioritas V1 > PROVISIONAL > BETA).

In [ ]:
import re
KOLEKSI = ['NISAR_L2_GCOV_V1', 'NISAR_L2_GCOV_PROVISIONAL_V1', 'NISAR_L2_GCOV_BETA_V1']
T_CARI = ('2026-05-01', '2026-11-15')

def daftar_granule(bbox, t0=T_CARI[0], t1=T_CARI[1]):
    rows = []
    for prio, sn in enumerate(KOLEKSI):
        try:
            hasil = earthaccess.search_data(short_name=sn, bounding_box=bbox, temporal=(t0, t1), count=2000)
        except Exception as ex:
            print(f'   {sn}: gagal ({ex})'); continue
        print(f'   {sn}: {len(hasil)} granule')
        for g in hasil:
            nama = g['umm']['GranuleUR']; b = nama.split('_')
            # NISAR_L2_PR_GCOV_<siklus>_<track>_<A/D>_<frame>_<mode>_<pol>_...
            url = next((u for u in g.data_links(access='external') if u.endswith('.h5')), None)
            tgl = re.search(r'_(\d{8})T\d{6}_', nama)
            if not (tgl and url and len(b) > 9): continue
            rows.append(dict(tanggal=f'{tgl.group(1)[:4]}-{tgl.group(1)[4:6]}-{tgl.group(1)[6:]}',
                             track=b[5], arah=b[6], frame=b[7], pol=b[9], koleksi=sn, prio=prio, nama=nama, url=url))
    df = pd.DataFrame(rows, columns=['tanggal','track','arah','frame','pol','koleksi','prio','nama','url'])
    return (df.sort_values('prio').drop_duplicates(['tanggal','track','arah','frame','pol'])
              .sort_values(['tanggal','track','frame']).reset_index(drop=True))

GRAN = {}
for k, v in AOI.items():
    print(f'\n=== {k} ===')
    df = daftar_granule(v['bbox']); GRAN[k] = df
    dh = df[df.pol.str.startswith(('DH', 'QP'))]
    if dh.empty: print('   tidak ada granule dual/quad-pol'); continue
    ring = (dh.groupby(['track','arah'])
              .agg(n_tanggal=('tanggal','nunique'), n_frame=('frame','nunique'),
                   awal=('tanggal','min'), akhir=('tanggal','max'),
                   tanggal=('tanggal', lambda s: ', '.join(sorted(set(s)))))
              .sort_values(['n_tanggal','n_frame'], ascending=False))
    print(ring.head(8).to_string())

## 9. Tetapkan lintasan (track + arah) dan tanggal
Dikelompokkan per **track + arah** (frame bersambung dari lintasan yang sama digabung per tanggal di sel 10).
Default: lintasan dengan tanggal terbanyak; pra = paling awal, pasca = paling akhir.
**Go/no-go: minimal 2 tanggal.** Idealnya pra SEBELUM puncak hotspot (lihat sel 6) dan pasca SESUDAHNYA.

In [ ]:
PILIH = {}
for k, df in GRAN.items():
    dh = df[df.pol.str.startswith(('DH', 'QP'))]
    if dh.empty: print(k, ': tidak ada granule dual/quad-pol — AOI ini dilewati'); continue
    n = dh.groupby(['track','arah']).agg(nt=('tanggal','nunique'), nf=('frame','nunique'))
    tr, ar = n.sort_values(['nt','nf'], ascending=False).index[0]
    g = dh[(dh.track == tr) & (dh.arah == ar)].copy()
    tgl = sorted(g.tanggal.unique())
    PILIH[k] = dict(granule=g, track=tr, arah=ar, pra=tgl[0], pasca=tgl[-1],
                    folder=f'{K}/mentah/{k}/gcov/T{tr}{ar}')
    status = 'OK' if len(tgl) >= 2 else 'TIDAK CUKUP (butuh >= 2 tanggal)'
    print(f'{k}: track {tr} {ar}, {len(tgl)} tanggal {tgl}, frame {sorted(g.frame.unique())} -> {status}')

# Override contoh (pilih lintasan/tanggal lain dari tabel sel 8):
# g = GRAN['riau']; g = g[(g.track == '040') & (g.arah == 'A') & g.pol.str.startswith('DH')]
# PILIH['riau'] = dict(granule=g, track='040', arah='A', pra='2026-07-11', pasca='2026-10-26', folder=f'{K}/mentah/riau/gcov/T040A')

## 10. Ambil potongan AOI — cepat (unduh paralel ke disk lokal → potong + multilook 80 m → hapus)
Kenapa sebelumnya lambat dan ~900 MB/tanggal: AOI besar disimpan di 20 m float32, streaming HTTPS membaca ribuan chunk kecil,
dan GeoTIFF ditulis langsung ke Drive (FUSE lambat). Sekarang:
- `MODE='unduh'`: granule utuh diunduh ke `/content` (disk lokal Colab, cepat), dipotong, lalu dihapus. Biasanya paling cepat
  karena AOI kita menutup sebagian besar frame. `MODE='stream'` untuk AOI kecil.
- `RESOLUSI_SUBSET=80`: multilook 4×4 (rata daya linear) saat memotong → ~50 MB/tanggal, bukan 900 MB.
- `PARALEL` tanggal diproses bersamaan. `HANYA_TANGGAL`: cukup pra..pasca (+ deret di antaranya).
- Berkas 20 m yang **sudah ada dilewati dan tetap dipakai** — pipeline merata-ratakannya ke 80 m saat dibaca.


In [ ]:
import shutil, time, concurrent.futures as cf
MODE = 'unduh'            # 'unduh' (disarankan) atau 'stream'
RESOLUSI_SUBSET = 80      # samakan dengan RESOLUSI di sel 11
PARALEL = 3
LOKAL = '/content/tmp_gcov'
os.makedirs(LOKAL, exist_ok=True)
print('disk lokal bebas: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

def potong_satu(k, r, tmp):
    if MODE == 'stream':
        fs = earthaccess.get_fsspec_https_session()
        with fs.open(r.url, cache_type='blockcache', block_size=16 * 1024 * 1024) as fo:
            return jb.subset_gcov(fo, AOI[k]['bbox'], tmp, f'f{r.frame}', resolusi_m=RESOLUSI_SUBSET)
    f = earthaccess.download([r.url], local_path=f'{tmp}/h5_{r.frame}', threads=4)[0]
    try:
        return jb.subset_gcov(str(f), AOI[k]['bbox'], tmp, f'f{r.frame}', resolusi_m=RESOLUSI_SUBSET)
    finally:
        shutil.rmtree(f'{tmp}/h5_{r.frame}', ignore_errors=True)   # granule besar langsung dihapus

def kerjakan_tanggal(k, tgl, rows):
    out = PILIH[k]['folder']; os.makedirs(out, exist_ok=True)
    if all(os.path.exists(f'{out}/{tgl}_{p}.tif') for p in ('hv', 'hh')):
        return f'{k} {tgl}: sudah ada, dilewati'
    t0 = time.time(); tmp = f'{LOKAL}/{k}_{tgl}'; os.makedirs(tmp, exist_ok=True)
    try:
        hasil = []
        for _, r in rows.iterrows():
            try: hasil.append(potong_satu(k, r, tmp))
            except ValueError as ex: print(f'   {tgl} frame {r.frame}: tidak beririsan AOI, dilewati')
        if not hasil: return f'{k} {tgl}: tidak ada frame yang beririsan AOI'
        for L, p in (('HVHV', 'hv'), ('HHHH', 'hh')):
            jb.gabung_frame([h.get(L) for h in hasil], f'{tmp}/{tgl}_{p}.tif')
            shutil.copyfile(f'{tmp}/{tgl}_{p}.tif', f'{out}/{tgl}_{p}.tif')
        return f'{k} {tgl}: {len(hasil)} frame digabung, {time.time()-t0:.0f} s'
    except Exception as ex:
        return f'{k} {tgl}: GAGAL {ex}'
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

for k, p in PILIH.items():
    g = p['granule']; g = g[(g.tanggal >= p['pra']) & (g.tanggal <= p['pasca'])]
    per_tgl = list(g.groupby('tanggal'))
    print(f'== {k}: track {p["track"]} {p["arah"]}, {len(per_tgl)} tanggal -> {p["folder"]}')
    with cf.ThreadPoolExecutor(PARALEL) as ex:
        for msg in ex.map(lambda t: kerjakan_tanggal(k, t[0], t[1]), per_tgl):
            print('  ', msg)

# cek cakupan: harus >= ~50% AOI
import rasterio, glob, numpy as np
for k, p in PILIH.items():
    w, s, e, n = AOI[k]['bbox']; luas_aoi = (e - w) * (n - s) * 111 * 111
    for f in sorted(glob.glob(f'{p["folder"]}/*_hv.tif')):
        with rasterio.open(f) as r:
            a = r.read(1); v = np.isfinite(a) & (a > 0)
            print(f'   {f.split("/")[-1]}: cakupan AOI ~{v.sum()*r.res[0]*r.res[1]/1e6/luas_aoi:.0%}')

## 11. Jalankan pipeline v3 per AOI
**Riau dulu sampai berhasil** (Checkpoint 1), lalu Kalteng, lalu Merauke (Checkpoint 2).

Ambang dipilih otomatis oleh Multi-Otsu pada superpixel SLIC di dalam KHG (dibatasi maksimal −1,8 dB).
Kalau histogram di sel 12 menunjukkan garis merah tidak di lembah, isi `AMBANG[k]` dengan angka (dB) — sebut di slide Validity.
`RESOLUSI` 80 m untuk percobaan pertama (cepat), 40 m kalau waktu cukup. Jangan menimpa `keluaran/` yang bagus dengan run gagal.

In [ ]:
AMBANG = {'riau': None, 'kalteng': None, 'merauke': None}   # None = Multi-Otsu otomatis
RESOLUSI = 80
JALANKAN = ['riau']          # tambah 'kalteng', 'merauke' setelah Riau beres

for k in JALANKAN:
    p = PILIH[k]; g = p.get('folder', f'{K}/mentah/{k}/gcov')
    deret = [(t, f'{g}/{t}_hv.tif', f'{g}/{t}_hh.tif') for t in sorted(p['granule'].tanggal.unique())
             if p['pra'] <= t <= p['pasca'] and os.path.exists(f'{g}/{t}_hv.tif')]
    row = lambda t: p['granule'][p['granule'].tanggal == t].iloc[0]
    aod = f'{K}/mentah/{k}/aod_merra2.tif'
    jb.proses(
        aoi_nama=AOI[k]['nama'], aoi_bbox=AOI[k]['bbox'],
        tanggal_pra=p['pra'], tanggal_pasca=p['pasca'],
        hv_pra=f'{g}/{p["pra"]}_hv.tif', hv_pasca=f'{g}/{p["pasca"]}_hv.tif',
        hh_pra=f'{g}/{p["pra"]}_hh.tif', hh_pasca=f'{g}/{p["pasca"]}_hh.tif',
        khg=f'{K}/mentah/{k}/khg.geojson', hotspot_csv=HOTSPOT,
        kabupaten=f'{K}/mentah/{k}/kabupaten.geojson', kolom_kabupaten=KOLOM_KAB,
        deret=deret, aod_tif=aod if os.path.exists(aod) else None, sumber_aod='MERRA-2 M2TMNXAER TOTEXTTAU',
        granule_pra=', '.join(p['granule'][p['granule'].tanggal == p['pra']].nama), granule_pasca=', '.join(p['granule'][p['granule'].tanggal == p['pasca']].nama),
        produk=row(p['pra'])['koleksi'], mode='nisar',
        folder_keluaran=f'{K}/keluaran/{k}', folder_olahan=f'{K}/olahan/{k}',
        param=jb.Parameter(ambang_hv_db=AMBANG[k], resolusi_m=RESOLUSI, ukuran_filter=3 if RESOLUSI >= 80 else 5),
    )

## 12. Periksa histogram Multi-Otsu dan plot asap–hotspot (keduanya untuk slide)

In [ ]:
from IPython.display import Image, display
for k in JALANKAN:
    print(k); display(Image(f'{K}/olahan/{k}/histogram_multiotsu.png')); display(Image(f'{K}/olahan/{k}/plot_asap_hotspot.png'))
    st = json.load(open(f'{K}/keluaran/{k}/statistik.json'))
    print({x: st[x] for x in ['total_ha','terkonfirmasi_ha','kandidat_tersembunyi_ha','perlu_verifikasi_ha',
                             'rasio_kebutaan_termal_akibat_asap_persen','arah_rambatan_dampak_dominan','ambang_dipakai_db']})

## 13. Tulis indeks dan bungkus untuk repo


In [ ]:
jb.tulis_indeks(f'{K}/keluaran')
assert all(a['mode'] == 'nisar' for a in json.load(open(f'{K}/keluaran/aoi.json'))['aoi'] if a['slug'] in JALANKAN)
shutil.make_archive('/content/keluaran', 'zip', f'{K}/keluaran')
shutil.make_archive('/content/olahan', 'zip', f'{K}/olahan')
from google.colab import files; files.download('/content/keluaran.zip'); files.download('/content/olahan.zip')
print('Ekstrak keluaran.zip menimpa folder keluaran/ di repo, lalu commit + push. olahan.zip untuk slide.')

---
**Periksa di QGIS sebelum commit:** buka `keluaran/<aoi>/terbakar.geojson` di atas `mentah/<aoi>/gcov/<pasca>_hv.tif`.
Poligon harus menempel di area gelap, bukan bergeser, dan tidak ada poligon di luar garis KHG.
